# Notebook 01 — Config, Runtime, and Time Axes

This notebook explores `config.toml`, `load_config()`, time-axis semantics,
interpolation modes, and `validate_config()`.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SAMPLE_DIR = PROJECT_ROOT / "notebooks" / "sample_data"
OUTPUT_DIR = PROJECT_ROOT / "notebooks" / "_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


## 1 · config.toml structure

PhosCrosstalk uses a TOML configuration file divided into sections:

| Section | Purpose |
|---------|---------|
| `[paths]` | Input/output file paths |
| `[model]` | Mechanism, scaling mode, length scale |
| `[optimisation]` | Solver, starts, loss type, regularisation |
| `[time]` | Time-point arrays for each modality |
| `[bounds]` | Min/max for each parameter class |
| `[solver]` | ODE solver settings (rtol, atol, …) |
| `[loss_weights]` | Per-modality loss coefficients |
| `[analysis]` | Steady-state, knockouts, sensitivity flags |
| `[neural_ode]` | Neural ODE refinement (optional) |
| `[pinn]` | PINN refinement (optional) |
| `[posterior]` | Bootstrap / profile likelihood (optional) |
| `[runtime]` | CPU thread budget |


In [ ]:
import pathlib
from phoscrosstalk.config import load_config

config_path = PROJECT_ROOT / "config.toml"
if config_path.exists():
    cfg = load_config(config_path)
    print("Loaded config.toml successfully")
    print("cfg.time:", cfg.time)
    print("cfg.model:", cfg.model)
else:
    print("config.toml not found — will build a minimal demo config below")


## 2 · Building a minimal config in memory

In [ ]:
import types

def make_ns(**kw):
    ns = types.SimpleNamespace()
    for k, v in kw.items():
        setattr(ns, k, v)
    return ns

# Minimal demo config — mirrors what load_config() returns
cfg_demo = types.SimpleNamespace(
    paths=make_ns(
        data="notebooks/sample_data/protephospho.csv",
        rna_data="notebooks/sample_data/mrna.csv",
        kinase_tsv="notebooks/sample_data/kinase_sites.tsv",
        tf_net="notebooks/sample_data/tf_mrna.csv",
        ptm_intra="",
        ptm_inter="",
        output_dir="notebooks/_outputs",
    ),
    model=make_ns(
        mechanism="dist",
        scale_mode="minmax",
        length_scale=50.0,
        weight_scheme="uniform",
    ),
    time=make_ns(
        phosphosite_time_points=list(range(1, 15)),
        mrna_time_points=list(range(1, 15)),
        protein_time_points=list(range(1, 15)),
        interpolation="linear",
    ),
    bounds=make_ns(
        rate_min=1e-5, rate_max=10.0,
        protein_degradation_max=0.5,
        k_deact_max=2.0,
        kinase_rate_max=3.0,
        phosphatase_rate_max=5.0,
        beta_coupling_max=3.0,
        alpha_min=0.01,
        gamma_abs_max=3.0,
    ),
    optimisation=make_ns(
        n_starts=8,
        max_steps=2000,
        ls_solver="lbfgsb",
        optimizer_backend="scipy",
        loss_type="mse",
        lambda_net=0.01,
        reg_lambda=0.001,
    ),
    solver=make_ns(
        ode_solver="Dopri5",
        ode_adjoint=False,
        rtol=1e-4,
        atol=1e-6,
        max_steps=16384,
        dt0=0.01,
    ),
    loss_weights=make_ns(phospho=1.0, abundance=0.5, mrna=0.25, reg=0.1),
    analysis=make_ns(
        tune=False,
        run_steadystate=True,
        run_knockouts=True,
        run_sensitivity=False,
    ),
    neural_ode=make_ns(enabled=False, width=64, depth=3, steps=500, learning_rate=1e-3),
    pinn=make_ns(enabled=False, width_size=64, depth=3, activation="tanh"),
    posterior=make_ns(enabled=False, method="bootstrap"),
    runtime=make_ns(cpu_threads=4, parallel_starts=4, threads_per_start=1),
)

print("Demo config built.  cfg_demo.time:")
print("  phosphosite_time_points:", cfg_demo.time.phosphosite_time_points)
print("  mrna_time_points       :", cfg_demo.time.mrna_time_points)
print("  protein_time_points    :", cfg_demo.time.protein_time_points)
print("  interpolation          :", cfg_demo.time.interpolation)


## 3 · Time axes — biological meaning

The time points are arbitrary ordinal indices (1–14 in the sample data) that
correspond to experimental collection times.  In a real experiment these might
be minutes post-stimulus (e.g. 0, 5, 10, 20, 30, 60, … min).

The framework keeps three *separate* time arrays because phosphosite, protein,
and mRNA measurements are often collected at different resolutions.


In [ ]:
t_phos = np.array(cfg_demo.time.phosphosite_time_points, dtype=float)
t_rna  = np.array(cfg_demo.time.mrna_time_points,        dtype=float)
t_prot = np.array(cfg_demo.time.protein_time_points,     dtype=float)

fig, ax = plt.subplots(figsize=(9, 2))
ax.scatter(t_phos, np.ones_like(t_phos)*2, s=40, label="phosphosite")
ax.scatter(t_prot, np.ones_like(t_prot)*1, s=40, marker="s", label="protein")
ax.scatter(t_rna,  np.zeros_like(t_rna),   s=40, marker="^", label="mRNA")
ax.set_yticks([0, 1, 2])
ax.set_yticklabels(["mRNA", "protein", "phosphosite"])
ax.set_xlabel("time index")
ax.set_title("Time axes for each data modality")
ax.legend(loc="upper left")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "01_time_axes.png", dpi=100)
plt.show()


## 4 · Interpolation modes

`cfg.time.interpolation` controls how `make_k_act_fn()` and `make_s_prod_fn()`
construct continuous-time driving signals from the discrete data:

| Mode | Behaviour |
|------|-----------|
| `"piecewise_constant"` | Step function — rate is held constant between sample times |
| `"linear"` | Linear interpolation — smooth ramp between adjacent samples |

`"linear"` is the default and gives smoother ODE trajectories.


In [ ]:
# Illustrate piecewise_constant vs linear interpolation of a hypothetical rate
t_demo = np.array([1., 2., 4., 8., 12.])
vals   = np.array([0.2, 0.8, 0.6, 0.4, 0.3])
t_fine = np.linspace(1, 12, 200)

# piecewise_constant
pc = np.array([vals[np.searchsorted(t_demo, ti, side='right') - 1] for ti in t_fine])
# linear
li = np.interp(t_fine, t_demo, vals)

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(t_fine, pc, label="piecewise_constant", drawstyle="steps-post")
ax.plot(t_fine, li, label="linear", linestyle="--")
ax.scatter(t_demo, vals, zorder=5, color="k", s=40)
ax.set_xlabel("time")
ax.set_ylabel("rate value")
ax.legend()
ax.set_title("Interpolation modes for driving signals")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "01_interpolation.png", dpi=100)
plt.show()


## 5 · Bounds section — log-space encoding

In [ ]:
bounds_df = pd.DataFrame({
    "parameter_class": [
        "k_deact (protein deactivation)",
        "d_deg (protein degradation)",
        "beta_g / beta_l (crosstalk coupling)",
        "alpha (kinase strength)",
        "kK_act / kK_deact (kinase rates)",
        "k_off (phosphatase rate)",
        "gamma (regulatory coupling)",
    ],
    "min (original scale)": [
        cfg_demo.bounds.rate_min,
        cfg_demo.bounds.rate_min,
        cfg_demo.bounds.rate_min,
        cfg_demo.bounds.alpha_min,
        cfg_demo.bounds.rate_min,
        cfg_demo.bounds.rate_min,
        -cfg_demo.bounds.gamma_abs_max,
    ],
    "max (original scale)": [
        cfg_demo.bounds.k_deact_max,
        cfg_demo.bounds.protein_degradation_max,
        cfg_demo.bounds.beta_coupling_max,
        cfg_demo.bounds.rate_max,
        cfg_demo.bounds.kinase_rate_max,
        cfg_demo.bounds.phosphatase_rate_max,
        cfg_demo.bounds.gamma_abs_max,
    ],
    "log-space?": ["Yes"]*6 + ["No"],
})
print(bounds_df.to_string(index=False))


## 6 · validate_config() walkthrough

In [ ]:
from phoscrosstalk.config import validate_config

# validate_config checks that all required fields exist and
# that referenced file paths are resolvable.
# We'll show what it checks rather than running it on the demo config
# (which uses relative paths that may not resolve here).

import inspect
src = inspect.getsource(validate_config)
# Print just the first 30 lines to show what is checked
for line in src.split("\n")[:35]:
    print(line)
